In [3]:
import cv2
import os
import torch
from ultralytics import YOLO
from pathlib import Path
from tqdm import tqdm

# ==========================================
# 1. Configuration
# ==========================================
# MODIFY HERE WITH YOUR VIDEO NAME
VIDEO_INPUT_NAME = "test_lyon.mp4" 

# Set paths
PROJECT_ROOT = Path(os.getcwd()).parent 
INPUT_DIR = Path(r"C:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project\data")        # Where you placed the video (e.g., project root)
OUTPUT_DIR = PROJECT_ROOT / "results" / "demo_videos"  # Where to save the output video
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_PATH = INPUT_DIR / VIDEO_INPUT_NAME
OUTPUT_PATH = OUTPUT_DIR / f"demo_{VIDEO_INPUT_NAME}"

# Hardware Configuration
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Computing device: {DEVICE}")

# ==========================================
# 2. MODEL LOADING
# ==========================================
# Set path to your custom trained model
custom_model_path = PROJECT_ROOT / 'runs/yolov8n_vehicle_detection2/weights/best.pt'

if custom_model_path.exists():
    print(f"Loading YOUR trained model: {custom_model_path}")
    model = YOLO(str(custom_model_path))
else:
    print("Custom model not found. Using standard pre-trained YOLOv8n.")
    model = YOLO("yolov8n.pt")

# ==========================================
# 3. VIDEO PROCESSING
# ==========================================
def process_custom_video():
    if not VIDEO_PATH.exists():
        print(f"Error: Video file not found: {VIDEO_PATH}")
        print("Make sure it is uploaded and the name is correct.")
        return

    # Open the video
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    
    # Read original video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Video opened: {width}x{height} @ {fps}fps ({total_frames} frames)")
    print(f"Saving to: {OUTPUT_PATH}")

    # Setup Video Writer
    # Use 'mp4v' for MP4
    writer = cv2.VideoWriter(str(OUTPUT_PATH), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    # Progress bar
    pbar = tqdm(total=total_frames, desc="Processing Video")

    while cap.isOpened():
        success, frame = cap.read() # Read a frame for each iteration
        if not success:
            break

        # --- INFERENCE ---
        # conf=0.25 discards weak detections
        # persist=True keeps stable IDs across frames
        # we use model.track() for tracking otherwise use model.predict() for detection only, but IDs won't be stable
        results = model.track(frame, persist=True, conf=0.25, verbose=False, device=DEVICE)[0]

        # --- DRAWING ---
        # The YOLO .plot() method automatically draws boxes, labels, and confidence
        # It's the cleanest way to show that the model is working.
        annotated_frame = results.plot()

        # Write frame
        writer.write(annotated_frame)
        pbar.update(1)

    # Release resources
    cap.release()
    writer.release()
    pbar.close()
    
    print("\n" + "="*50)
    print(" PROCESSING COMPLETED!")
    print(f" The demo video is ready: {OUTPUT_PATH}")
    print("="*50)

# Start the script
process_custom_video()

Computing device: cuda
Loading YOUR trained model: c:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project\runs\yolov8n_vehicle_detection2\weights\best.pt
Video opened: 480x848 @ 29fps (1815 frames)
Saving to: c:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project\results\demo_videos\demo_test_lyon.mp4


Processing Video: 100%|██████████| 1815/1815 [01:13<00:00, 24.64it/s]


 PROCESSING COMPLETED!
 The demo video is ready: c:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project\results\demo_videos\demo_test_lyon.mp4
